<a href="https://colab.research.google.com/github/Mohamed-Hesham-Latif/ML-55-25005/blob/main/Task_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 3

Due to the simplicity of KNN for Classification, let's focus on using a Pipeline and a GridSearchCV tool, since these skills can be generalized for any model.


## The Sonar Data

### Detecting a Rock or a Mine

Sonar (sound navigation ranging) is a technique that uses sound propagation (usually underwater, as in submarine navigation) to navigate, communicate with or detect objects on or under the surface of the water, such as other vessels.



The data set contains the response metrics for 60 separate sonar frequencies sent out against a known mine field (and known rocks). These frequencies are then labeled with the known object they were beaming the sound at (either a rock or a mine).



Our main goal is to create a machine learning model capable of detecting the difference between a rock or a mine based on the response of the 60 separate sonar frequencies.


Data Source: https://archive.ics.uci.edu/ml/datasets/Connectionist+Bench+(Sonar,+Mines+vs.+Rocks)

### Complete the Tasks in bold

**TASK: Run the cells below to load the data.**

In [2]:
import numpy as np
import pandas as pd

In [1]:
import numpy as np
import pandas as pd

# URL of the UCI Sonar dataset
# This dataset contains sonar signals bounced off metal cylinders (mines) or rocks
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/undocumented/connectionist-bench/sonar/sonar.all-data"

# Read the dataset into a pandas DataFrame
# header=None is used because the dataset file does not contain column headers
df = pd.read_csv(url, header=None)

# Separate the dataset into features (X) and labels (y)

# X contains the first 60 columns (0 to 59)
# These columns represent numeric sonar signal measurements
X = df.iloc[:, :-1]

# y contains the last column (column 60)
# This column represents the class label:
# 'R' = Rock
# 'M' = Mine (metal cylinder)
y = df.iloc[:, -1]

# Display the first 5 rows of the dataset
# This helps verify the data was loaded correctly
df.head()

,0,1,2,3,4,5,6,7,8,9,...,51,52,53,54,55,56,57,58,59,60
0,0.0200,0.0371,0.0428,0.0207,0.0954,0.0986,0.1539,0.1601,0.3109,0.2111,...,0.0027,0.0065,0.0159,0.0072,0.0167,0.0180,0.0084,0.0090,0.0032,R
1,0.0453,0.0523,0.0843,0.0689,0.1183,0.2583,0.2156,0.3481,0.3337,0.2872,...,0.0084,0.0089,0.0048,0.0094,0.0191,0.0140,0.0049,0.0052,0.0044,R
2,0.0262,0.0582,0.1099,0.1083,0.0974,0.2280,0.2431,0.3771,0.5598,0.6194,...,0.0232,0.0166,0.0095,0.0180,0.0244,0.0316,0.0164,0.0095,0.0078,R
3,0.0100,0.0171,0.0623,0.0205,0.0205,0.0368,0.1098,0.1276,0.0598,0.1264,...,0.0121,0.0036,0.0150,0.0085,0.0073,0.0050,0.0044,0.0040,0.0117,R
4,0.0762,0.0666,0.0481,0.0394,0.0590,0.0649,0.1209,0.2467,0.3564,0.4459,...,0.0031,0.0054,0.0105,0.0110,0.0015,0.0072,0.0048,0.0107,0.0094,R


## Train | Test Split

Our approach here will be one of using Cross Validation on 90% of the dataset, and then judging our results on a final test set of 10% to evaluate our model.

**TASK: Split the data into features and labels, and then split into a training set and test set, with 90% for Cross-Validation training, and 10% for a final test set.**

*Note: The solution uses a random_state=42*

In [3]:
# Select all rows and all columns except the last one
# These columns represent the input features (60 sonar signal measurements)
X = df.iloc[:, :-1]

# Select all rows from the last column
# This column contains the target labels:
# 'R' = Rock, 'M' = Mine
y = df.iloc[:, -1]

In [4]:
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets
# X_train, y_train -> data used to train the machine learning model
# X_test, y_test -> data used to evaluate the model's performance

X_train, X_test, y_train, y_test = train_test_split(
    X, y,                 # Features (X) and labels (y)
    test_size=0.1,        # 10% of the data will be used for testing, 90% for training
    random_state=42       # Ensures the split is reproducible (same result every run)
)

**TASK: Create a Pipeline that contains both a StandardScaler and a KNN model**

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# Create a machine learning pipeline
# A pipeline allows multiple preprocessing and modeling steps
# to be executed sequentially as a single workflow
pipe = Pipeline([

    # Step 1: Standardize the feature values
    # StandardScaler transforms the data so that each feature
    # has mean = 0 and standard deviation = 1
    # This is important for KNN because it relies on distance calculations
    ('scaler', StandardScaler()),

    # Step 2: Apply the K-Nearest Neighbors classifier
    # KNN predicts the class based on the majority label
    # among the k closest data points
    ('knn', KNeighborsClassifier())
])

**TASK: Perform a grid-search with the pipeline to test various values of k and report back the best performing parameters.**

In [6]:
# Define the hyperparameter grid for tuning the KNN model
# These parameters will be tested during Grid Search

param_grid = {
    # Number of neighbors (k) to consider in KNN
    # range(3, 41, 2) generates odd numbers from 3 to 39
    # Odd numbers help avoid ties in classification
    "knn__n_neighbors": list(range(3, 41, 2)),   # 3,5,7,...,39

    # Weighting strategy for neighbors
    # "uniform"  -> all neighbors contribute equally
    # "distance" -> closer neighbors have more influence
    "knn__weights": ["uniform", "distance"]
}

In [7]:
from sklearn.model_selection import GridSearchCV

# Create a GridSearchCV object to find the best hyperparameters
# It tests different parameter combinations from param_grid
# using cross-validation

grid = GridSearchCV(
    pipe,        # The pipeline (scaling + KNN model)
    param_grid,  # The grid of hyperparameters to search through
    cv=10        # Use 10-fold cross-validation to evaluate each combination
)

# Train the model and perform the grid search
# This fits the pipeline multiple times with different parameter combinations
# and selects the one that gives the best cross-validation performance
grid.fit(X_train, y_train)

GridSearchCV(cv=10,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('knn', KNeighborsClassifier())]),
             param_grid={'knn__n_neighbors': [3, 5, 7, 9, 11, 13, 15, 17, 19,
                                              21, 23, 25, 27, 29, 31, 33, 35,
                                              37, 39],
                         'knn__weights': ['uniform', 'distance']})

In [8]:
# Print the best hyperparameter combination found during Grid Search
# This shows the values of n_neighbors and weights that produced the best results
print("Best Parameters:", grid.best_params_)

# Print the best cross-validation score achieved with those parameters
# This score is the average accuracy across the 10 cross-validation folds
print("Best CV Score:", grid.best_score_)

Best Parameters: {'knn__n_neighbors': 3, 'knn__weights': 'uniform'}
Best CV Score: 0.8181286549707603


### Final Model Evaluation

**TASK: Using the grid classifier object from the previous step, get a final performance classification report and confusion matrix.**

In [9]:
# Use the trained GridSearchCV model to make predictions on the test dataset
# grid.predict() automatically uses the best model found during the grid search
# X_test contains the unseen data used to evaluate the model's performance
y_pred = grid.predict(X_test)

In [10]:
from sklearn.metrics import classification_report

# Generate and display a classification report
# This compares the true labels (y_test) with the predicted labels (y_pred)

# The report includes:
# - Precision: How many predicted positives are actually correct
# - Recall: How many actual positives were correctly identified
# - F1-score: Harmonic mean of precision and recall
# - Support: Number of true samples for each class

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           M       1.00      0.92      0.96        13
           R       0.89      1.00      0.94         8

    accuracy                           0.95        21
   macro avg       0.94      0.96      0.95        21
weighted avg       0.96      0.95      0.95        21



In [11]:
from sklearn.metrics import confusion_matrix

# Generate the confusion matrix to evaluate classification performance
# It compares the actual labels (y_test) with the predicted labels (y_pred)

# The matrix shows:
# - True Positives
# - True Negatives
# - False Positives
# - False Negatives

print(confusion_matrix(y_test, y_pred))

[[12  1]
 [ 0  8]]
